# ДЗ2 — нагрузочное испытание и профилирование

Дата запуска: 2026-11-20. Испытания выполняются на коде сервиса из HEAD `f7cd4618c4b28d66f8aec892f22cf906b0e5b980`; начальный `git status --short` до добавления материалов ДЗ2 был пуст. Критерии приемки: `p95 < 500 ms` и доля ошибок `< 1%`. Сценарий и веса смешанного профиля фиксированы до запуска: список 35%, карточка 25%, выдача 20%, возврат 15%, сводка 5%.

## Порядок
0. Окружение, версия кода и состояние репозитория  
1. Инструменты k6 и py-spy  
2. PostgreSQL, миграции и идентификаторы для нагрузки (только SELECT)  
3. Запуск API и короткая проверка сценариев  
4. Ступени смешанного профиля с рядами ресурсов  
5. Проверка целостности и анализ mixed  
6. Ступени тяжелой операции `GET /api/v1/summary`  
7. Профиль py-spy тяжелой операции  
8. EXPLAIN ANALYZE гипотез и проверка данных  
9. Графики, вердикты и итоговый список файлов

## Шаг 0. Условия испытания и версия кода
Собираются ОС, CPU/ядра/потоки, память, версии Docker/PostgreSQL/Python/k6/py-spy, HEAD и `git status --short`. Полный файл сохраняется в `logs/environment.txt`. Исходное состояние рабочей копии до создания файлов этого задания было чистым; появившиеся далее изменения относятся к самому сценарию и ноутбуку ДЗ2.

In [1]:
%%bash
cd "$(git rev-parse --show-toplevel)"
set -euo pipefail
mkdir -p logs/dz2 reports profiles
RUN_DIR="logs/dz2/$(date -u +%Y%m%dT%H%M%SZ)"
mkdir -p "$RUN_DIR"/{mixed,heavy,resources,smoke,ids}
printf '%s\n' "$RUN_DIR" > logs/dz2/current_run.txt
export PATH="$HOME/.local/bin:$PATH"
{
  echo 'DZ2 environment'; date -Is; echo
  echo '--- uname ---'; uname -a; echo
  echo '--- lscpu ---'; lscpu; echo
  echo '--- memory ---'; free -h; echo
  echo '--- Python ---'; python3 --version; echo
  echo '--- Docker ---'; docker --version; docker compose version; echo
  echo '--- PostgreSQL client/server ---'; psql --version; docker compose exec -T ivan_bashkin_postgres psql -X -A -t -U app -d ivan_bashkin -c 'SELECT version();' </dev/null; echo
  echo '--- k6 ---'; k6 version; echo
  echo '--- py-spy ---'; .venv/bin/py-spy --version; echo
  echo '--- Git HEAD ---'; git rev-parse HEAD; echo
  echo '--- Git status at measurement preparation ---'; git status --short; echo
  echo '--- Initial pre-task status ---'; echo 'clean (verified before adding load/library.js and this notebook)'
} | tee logs/environment.txt
echo "Run directory: $RUN_DIR"

DZ2 environment


2026-11-20T17:58:41-03:00

--- uname ---


Linux sxsewvn 7.3.4-arch1-2 #1 SMP PREEMPT_DYNAMIC Thu, 12 Nov 2026 20:11:37 +0000 x86_64 GNU/Linux



--- lscpu ---


Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-

bit
Address sizes:                           48 bits physical, 48 bits virtual
Byte Order:          

... (50 outputs left out)


--- Git status at measurement preparation ---


 M README.md
?? load/
?? logs/dz2/
?? logs/environment.txt
?? notebooks/dz2_run.ipynb
?? reports/



--- Initial pre-task status ---
clean (verified before adding load/library.js and this notebook)


Run directory: logs/dz2/20261120T205841Z


## Шаг 1. Инструменты
При необходимости k6 устанавливается официальной Go-командой в `~/.local/bin`; py-spy и matplotlib устанавливаются только в `.venv`. Вывод фиксирует фактические версии и доступность ptrace-профилировщика.

In [2]:
%%bash
cd "$(git rev-parse --show-toplevel)"
set -euo pipefail
mkdir -p "$HOME/.local/bin"
if [ ! -x "$HOME/.local/bin/k6" ] && ! command -v k6 >/dev/null 2>&1; then
  command -v go >/dev/null 2>&1 || { echo 'k6 отсутствует, Go не установлен; остановка.' >&2; exit 1; }
  GOBIN="$HOME/.local/bin" go install go.k6.io/k6@latest
fi
export PATH="$HOME/.local/bin:$PATH"
if [ ! -x .venv/bin/py-spy ]; then .venv/bin/pip install py-spy; fi
if ! .venv/bin/python -c 'import matplotlib' >/dev/null 2>&1; then .venv/bin/pip install matplotlib; fi
k6 version
.venv/bin/py-spy --version
.venv/bin/python -c 'import matplotlib; print("matplotlib", matplotlib.__version__)'

k6 v1.8.3 (go1.27.4-X:nodwarf5, linux/amd64)


py-spy 0.4.4


matplotlib 3.11.2


## Шаг 2. PostgreSQL, миграции и пул идентификаторов
Поднимается PostgreSQL и применяется текущая миграция. После наполнения large запросами в `BEGIN READ ONLY` выгружаются JSON-пулы: свободные экземпляры с читателем без активной просрочки, активные выдачи для возврата, ID выдач для карточек. Счётчики JSON-пулов будут проверены в следующей ячейке.

In [3]:
%%bash
cd "$(git rev-parse --show-toplevel)"
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
RUN_DIR=$(cat logs/dz2/current_run.txt)
docker compose up -d
ready=0
for attempt in $(seq 1 30); do
  echo "PostgreSQL readiness ${attempt}/30"
  if docker compose exec -T ivan_bashkin_postgres pg_isready -U app -d ivan_bashkin </dev/null; then ready=1; break; fi
  sleep 2
done
if [ "$ready" -ne 1 ]; then echo 'PostgreSQL не готов за 60 секунд' >&2; exit 1; fi
.venv/bin/alembic upgrade head
.venv/bin/python seed.py --type=large | sed -E 's#(Login: admin / ).*#\1[REDACTED]#'
docker compose exec -T ivan_bashkin_postgres psql -X -qAt -v ON_ERROR_STOP=1 -U app -d ivan_bashkin -c "BEGIN READ ONLY; SELECT COALESCE(json_agg(json_build_object('reader_id', pool.reader_id, 'copy_id', pool.copy_id) ORDER BY pool.copy_id), '[]'::json) FROM (SELECT (SELECT r.id FROM readers r WHERE NOT EXISTS (SELECT 1 FROM loans l WHERE l.reader_id = r.id AND l.return_date IS NULL AND l.due_date < now()) ORDER BY r.id LIMIT 1) AS reader_id, bc.id AS copy_id FROM book_copies bc WHERE bc.status = 'available' ORDER BY bc.id LIMIT 22500) pool; COMMIT;" </dev/null > "$RUN_DIR/ids/issue_pairs.json"
docker compose exec -T ivan_bashkin_postgres psql -X -qAt -v ON_ERROR_STOP=1 -U app -d ivan_bashkin -c "BEGIN READ ONLY; SELECT COALESCE(json_agg(id ORDER BY id), '[]'::json) FROM (SELECT id FROM loans WHERE return_date IS NULL ORDER BY id LIMIT 7500) active_loans; COMMIT;" </dev/null > "$RUN_DIR/ids/return_loan_ids.json"
docker compose exec -T ivan_bashkin_postgres psql -X -qAt -v ON_ERROR_STOP=1 -U app -d ivan_bashkin -c "BEGIN READ ONLY; SELECT COALESCE(json_agg(id ORDER BY id), '[]'::json) FROM (SELECT id FROM loans ORDER BY id) all_loans; COMMIT;" </dev/null > "$RUN_DIR/ids/detail_loan_ids.json"
echo "ID files saved under $RUN_DIR/ids (contents intentionally not dumped)."

 Container ivan_bashkin_postgres Running 


PostgreSQL readiness 1/30


/var/run/postgresql:5432 - accepting connections


INFO  [alembic.runtime.migration] Context impl PostgresqlImpl.
INFO  [alembic.runtime.migration] Con

text impl PostgresqlImpl.
INFO  [alembic.runtime.migration] Will assume transactional DDL.
INFO  [al

embic.runtime.migration] Will assume transactional DDL.


Seeded 3000 readers, 10000 books, 30000 copies, 75000 loans. Login: admin / [REDACTED]


ID files saved under logs/dz2/20261120T205841Z/ids (contents intentionally not dumped).


### Проверка пулов и доступной ступенчатой нагрузки
Python-ячейка проверяет именно фактически сформированные файлы. Условия пула рассчитаны на всю заранее определённую ladder до 512 req/s; ошибки по причине отсутствия ID будут исключены.

In [4]:
import json
from pathlib import Path

repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "logs/dz2/current_run.txt").is_file())
run_dir = repo_root / Path((repo_root / "logs/dz2/current_run.txt").read_text().strip())
issue_pairs = json.loads((run_dir / "ids/issue_pairs.json").read_text())
return_ids = json.loads((run_dir / "ids/return_loan_ids.json").read_text())
detail_ids = json.loads((run_dir / "ids/detail_loan_ids.json").read_text())
rates = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
max_iterations = 30 * sum(rates)
max_issue_slots = (max_iterations // 100) * 20 + 20
max_return_slots = (max_iterations // 100) * 15 + 15
assert issue_pairs and issue_pairs[0]["reader_id"] is not None
assert len(issue_pairs) >= max_issue_slots, (len(issue_pairs), max_issue_slots)
assert len(return_ids) >= max_return_slots, (len(return_ids), max_return_slots)
assert len(detail_ids) >= max_iterations
print(f"issue_pairs: {len(issue_pairs)} (потребность для всех запросов смешанного теста: не более {max_issue_slots})")
print(f"active return IDs: {len(return_ids)} (потребность: не более {max_return_slots})")
print(f"detail IDs: {len(detail_ids)} (потребность: не более {max_iterations})")
print("Пулов достаточно для запланированной лестницы 1..512 req/s.")

issue_pairs: 22500 (потребность для всех запросов смешанного теста: не более 6140)
active return IDs: 7500 (потребность: не более 4605)
detail IDs: 75000 (потребность: не более 30690)
Пулов достаточно для запланированной лестницы 1..512 req/s.


## Шаг 3. API и короткая проверка k6
Запускается один worker uvicorn. Mixed и heavy проверяются короткими ступенями по 3 секунды; эти результаты — только smoke-тест и не попадут в итоговые таблицы ДЗ2. После smoke mixed база будет заново заполнена перед испытаниями.

In [5]:
%%bash
cd "$(git rev-parse --show-toplevel)"
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
RUN_DIR=$(cat logs/dz2/current_run.txt)
mkdir -p logs
nohup .venv/bin/uvicorn app.main:app --port 8000 > "$RUN_DIR/uvicorn.log" 2>&1 &
echo $! > "$RUN_DIR/uvicorn.pid"
ready=0
for attempt in $(seq 1 30); do
  echo "API readiness ${attempt}/30"
  if curl --fail --silent --max-time 2 http://localhost:8000/docs >/dev/null; then ready=1; break; fi
  sleep 1
done
if [ "$ready" -ne 1 ]; then cat "$RUN_DIR/uvicorn.log"; kill "$(cat "$RUN_DIR/uvicorn.pid")" 2>/dev/null || true; exit 1; fi
k6 run -e SCENARIO=mixed -e RATE=1 -e DURATION=3s -e OFFSET=0 -e DATA_DIR="$PWD/$RUN_DIR/ids" --summary-export "$RUN_DIR/smoke/mixed.json" load/library.js > "$RUN_DIR/smoke/mixed.log" 2>&1
k6 run -e SCENARIO=heavy -e RATE=1 -e DURATION=3s -e DATA_DIR="$PWD/$RUN_DIR/ids" --summary-export "$RUN_DIR/smoke/heavy.json" load/library.js > "$RUN_DIR/smoke/heavy.log" 2>&1
echo '--- Mixed smoke output ---'; cat "$RUN_DIR/smoke/mixed.log"
echo '--- Heavy smoke output ---'; cat "$RUN_DIR/smoke/heavy.log"

API readiness 1/30


API readiness 2/30


--- Mixed smoke output ---



         /\      Grafana   /‾‾/  
    /\  /  \     |\  __   /  /   
   /  \/    \    | |/ /  / 

  ‾‾\ 
  /          \   |   (  |  (‾)  |
 / __________ \  |_|\_\  \_____/ 


     execution: l

ocal
        script: load/library.js
        output: -

     scenarios: (100.00%) 1 scenario, 20 max

... (55 outputs left out)

....................: 3      0.981751/s
    vus............................: 0      min=0      max=0

 
    vus_max........................: 10     min=10     max=10

    NETWORK
    data_received......

............: 3.5 kB 1.2 kB/s
    data_sent......................: 870 B  285 B/s




running (03.1s

), 00/10 VUs, 3 complete and 0 interrupted iterations
heavy ✓ [ 100% ] 00/10 VUs  3s  1.00 iters/s

## Шаг 4. Испытание смешанным профилем
Перед испытанием large заново генерируется и ID-пулы повторно извлекаются read-only запросами. Запускаются ступени 1, 2, 4, 8, 16, 32, 64, 128, 256, 512 req/s по 30 секунд; лестница прекращается на первой ступени, где k6 возвращает threshold exit (p95 ≥ 500 ms или ошибки ≥ 1%). Между успешными ступенями CPU приложения должен быть ниже 5% одного ядра пять последовательных секунд. `--summary-export` и отдельный CSV ресурсов сохраняют полные результаты.

In [6]:
%%bash
cd "$(git rev-parse --show-toplevel)"
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
RUN_DIR=$(cat logs/dz2/current_run.txt)
.venv/bin/python seed.py --type=large | sed -E 's#(Login: admin / ).*#\1[REDACTED]#'
docker compose exec -T ivan_bashkin_postgres psql -X -qAt -v ON_ERROR_STOP=1 -U app -d ivan_bashkin -c "BEGIN READ ONLY; SELECT COALESCE(json_agg(json_build_object('reader_id', pool.reader_id, 'copy_id', pool.copy_id) ORDER BY pool.copy_id), '[]'::json) FROM (SELECT (SELECT r.id FROM readers r WHERE NOT EXISTS (SELECT 1 FROM loans l WHERE l.reader_id = r.id AND l.return_date IS NULL AND l.due_date < now()) ORDER BY r.id LIMIT 1) AS reader_id, bc.id AS copy_id FROM book_copies bc WHERE bc.status = 'available' ORDER BY bc.id LIMIT 22500) pool; COMMIT;" </dev/null > "$RUN_DIR/ids/issue_pairs.json"
docker compose exec -T ivan_bashkin_postgres psql -X -qAt -v ON_ERROR_STOP=1 -U app -d ivan_bashkin -c "BEGIN READ ONLY; SELECT COALESCE(json_agg(id ORDER BY id), '[]'::json) FROM (SELECT id FROM loans WHERE return_date IS NULL ORDER BY id LIMIT 7500) active_loans; COMMIT;" </dev/null > "$RUN_DIR/ids/return_loan_ids.json"
docker compose exec -T ivan_bashkin_postgres psql -X -qAt -v ON_ERROR_STOP=1 -U app -d ivan_bashkin -c "BEGIN READ ONLY; SELECT COALESCE(json_agg(id ORDER BY id), '[]'::json) FROM (SELECT id FROM loans ORDER BY id) all_loans; COMMIT;" </dev/null > "$RUN_DIR/ids/detail_loan_ids.json"
UVICORN_PID=$(cat "$RUN_DIR/uvicorn.pid")
read_ticks() { awk '{sub(/^.*\) /, ""); print $12 + $13}' "/proc/$UVICORN_PID/stat"; }
CLK_TCK=$(getconf CLK_TCK)
offset=0
last_rate=0
break_rate=none
for rate in 1 2 4 8 16 32 64 128 256 512; do
  echo "=== mixed: RATE=$rate req/s, DURATION=30s, OFFSET=$offset ==="
  json="$RUN_DIR/mixed/mixed_${rate}.json"
  log="$RUN_DIR/mixed/mixed_${rate}.log"
  resource="$RUN_DIR/resources/mixed_${rate}.csv"
  stopfile="$RUN_DIR/resources/stop_mixed_${rate}"
  rm -f "$stopfile"
  printf 'timestamp,phase,rate,app_cpu_pct_one_core,ps_avg_cpu_pct,app_rss_kb,pg_cpu_pct,pg_mem_usage,db_connections,db_active_connections\n' > "$resource"
  (
    previous=$(read_ticks)
    while [ ! -e "$stopfile" ]; do
      sleep 1
      current=$(read_ticks)
      app_cpu=$(awk -v d="$((current-previous))" -v hz="$CLK_TCK" 'BEGIN { printf "%.2f", d*100/hz }')
      previous=$current
      ps_values=$(ps -p "$UVICORN_PID" -o %cpu=,rss= | awk '{printf "%s,%s", $1, $2}')
      pg_stats=$(docker stats --no-stream --format '{{.CPUPerc}},{{.MemUsage}}' ivan_bashkin_postgres | awk -F, '{gsub(/%/, "", $1); gsub(/ /, "", $2); printf "%s,%s", $1, $2}')
      db_conn=$(docker compose exec -T ivan_bashkin_postgres psql -X -qAt -U app -d ivan_bashkin -c "SELECT count(*) || ',' || count(*) FILTER (WHERE state='active') FROM pg_stat_activity WHERE datname=current_database()" </dev/null | tr -d '\r\n')
      printf '%s,mixed_%s,%s,%s,%s,%s,%s\n' "$(date -Is)" "$rate" "$rate" "$app_cpu" "$ps_values" "$pg_stats" "$db_conn" >> "$resource"
    done
  ) &
  monitor_pid=$!
  set +e
  k6 run -e SCENARIO=mixed -e RATE="$rate" -e DURATION=30s -e OFFSET="$offset" -e DATA_DIR="$PWD/$RUN_DIR/ids" --summary-export "$json" load/library.js > "$log" 2>&1
  k6_status=$?
  set -e
  if [ ! -f "$json" ]; then touch "$stopfile"; wait "$monitor_pid" || true; cat "$log"; exit 1; fi
  echo '--- Full k6 output (also saved in JSON/log) ---'; cat "$log"
  echo "k6 exit status: $k6_status"
  offset=$((offset + rate * 30))
  if [ "$k6_status" -eq 99 ]; then
    break_rate=$rate
    touch "$stopfile"; wait "$monitor_pid"
    echo "Threshold violated at mixed rate $rate; stopping at first failing step."
    break
  elif [ "$k6_status" -ne 0 ]; then
    touch "$stopfile"; wait "$monitor_pid" || true
    echo "k6 failed unexpectedly (status $k6_status); stopping." >&2
    exit "$k6_status"
  fi
  stable=0
  previous=$(read_ticks)
  for second in $(seq 1 180); do
    sleep 1
    current=$(read_ticks)
    idle_cpu=$(awk -v d="$((current-previous))" -v hz="$CLK_TCK" 'BEGIN { printf "%.2f", d*100/hz }')
    previous=$current
    if awk -v cpu="$idle_cpu" 'BEGIN { exit !(cpu < 5) }'; then stable=$((stable+1)); else stable=0; fi
    if [ "$stable" -ge 5 ]; then echo "CPU idle recovered: <5% for 5 seconds (after ${second}s)."; break; fi
  done
  touch "$stopfile"; wait "$monitor_pid"
  if [ "$stable" -lt 5 ]; then echo 'CPU did not return to idle within 180 seconds; stop before next stage.' >&2; exit 1; fi
  last_rate=$rate
done
printf '%s\n' "$break_rate" > "$RUN_DIR/mixed/break_rate.txt"
printf '%s\n' "$last_rate" > "$RUN_DIR/mixed/last_passed_rate.txt"
echo "Mixed staircase finished; first failed rate=$break_rate, last passing rate=$last_rate."

Seeded 3000 readers, 10000 books, 30000 copies, 75000 loans. Login: admin / [REDACTED]


=== mixed: RATE=1 req/s, DURATION=30s, OFFSET=0 ===


--- Full k6 output (also saved in JSON/log) ---



         /\      Grafana   /‾‾/  
    /\  /  \     |\  __   /  /   
   /  \/    \    | |/ /  / 

  ‾‾\ 
  /          \   |   (  |  (‾)  |
 / __________ \  |_|\_\  \_____/ 


     execution: l

ocal
        script: load/library.js
        output: -

     scenarios: (100.00%) 1 scenario, 20 max

... (992 outputs left out)

req_duration, http_req_failed' have been crossed"


k6 exit status: 99


Threshold violated at mixed rate 256; stopping at first failing step.


Mixed staircase finished; first failed rate=256, last passing rate=128.


## Шаг 5. Mixed: проверка одновременной выдачи и таблица ступеней
Read-only SQL проверяет отсутствие двух активных выдач одного экземпляра. Python строит таблицу только по JSON summary k6, выводит фактическую частоту/p50/p95/max/errors/dropped iterations, создаёт CSV и график p95 с порогом 500 ms. Для ступени излома дополнительно агрегируются реальные resource samples и отдельные latency/DB/app метрики каждой операции.

In [7]:
%%bash
cd "$(git rev-parse --show-toplevel)"
set -euo pipefail
RUN_DIR=$(cat logs/dz2/current_run.txt)
docker compose exec -T ivan_bashkin_postgres psql -X -qAt -v ON_ERROR_STOP=1 -U app -d ivan_bashkin -c "BEGIN READ ONLY; SELECT copy_id, count(*) FROM loans WHERE return_date IS NULL GROUP BY copy_id HAVING count(*) > 1; COMMIT;" </dev/null > "$RUN_DIR/mixed/duplicate_active_copies.txt"
if [ -s "$RUN_DIR/mixed/duplicate_active_copies.txt" ]; then echo 'Duplicate active copies detected:' >&2; cat "$RUN_DIR/mixed/duplicate_active_copies.txt"; exit 1; fi
echo 'No duplicate active copy assignments (query returned zero rows).' | tee "$RUN_DIR/mixed/duplicate_active_copies.txt"

No duplicate active copy assignments (query returned zero rows).


In [8]:
import csv
import json
from pathlib import Path
import statistics
import matplotlib.pyplot as plt

repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "logs/dz2/current_run.txt").is_file())
run_dir = repo_root / Path((repo_root / "logs/dz2/current_run.txt").read_text().strip())
def metric_values(metrics, name):
    metric = metrics.get(name, {})
    return metric.get("values", metric)

def read_stages(kind):
    rows = []
    for summary_path in sorted((run_dir / kind).glob(f"{kind}_*.json"), key=lambda p: int(p.stem.rsplit("_", 1)[1])):
        rate = int(summary_path.stem.rsplit("_", 1)[1])
        data = json.loads(summary_path.read_text())
        metrics = data["metrics"]
        duration = metric_values(metrics, "http_req_duration")
        failed = metric_values(metrics, "http_req_failed")
        requests = metric_values(metrics, "http_reqs")
        dropped = metric_values(metrics, "dropped_iterations").get("count", 0)
        db_values = metric_values(metrics, "db_time_ms")
        app_values = metric_values(metrics, "app_time_ms")
        db_share_values = metric_values(metrics, "db_time_share_pct")
        query_values = metric_values(metrics, "db_query_count")
        request_count = requests.get("count", 0)
        if request_count == 0:
            raise ValueError(f"No HTTP requests recorded for {summary_path}")
        rows.append({"rate": rate, "actual_rate": requests.get("rate", 0), "count": request_count, "median_ms": duration.get("med", 0), "p95_ms": duration.get("p(95)", 0), "max_ms": duration.get("max", 0), "error_rate": failed.get("value", failed.get("rate", 0)), "dropped_iterations": dropped, "db_median_ms": db_values.get("med", 0), "db_p95_ms": db_values.get("p(95)", 0), "app_median_ms": app_values.get("med", 0), "app_p95_ms": app_values.get("p(95)", 0), "db_time_share_median_pct": db_share_values.get("med", 0), "db_queries_per_request": query_values.get("count", 0) / request_count})
    return rows

mixed_rows = read_stages("mixed")
with (run_dir / "mixed/stages.csv").open("w", newline="", encoding="utf-8") as target:
    writer = csv.DictWriter(target, fieldnames=["rate", "actual_rate", "count", "median_ms", "p95_ms", "max_ms", "error_rate", "dropped_iterations", "db_median_ms", "db_p95_ms", "app_median_ms", "app_p95_ms", "db_time_share_median_pct", "db_queries_per_request"])
    writer.writeheader(); writer.writerows(mixed_rows)
print("| Задано req/s | Фактически req/s | Медиана ms | p95 ms | Максимум ms | Ошибки % | dropped_iterations | DB median ms | app median ms | DB доля median % | DB queries/request |\n|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
for row in mixed_rows:
    print(f"| {row['rate']} | {row['actual_rate']:.2f} | {row['median_ms']:.2f} | {row['p95_ms']:.2f} | {row['max_ms']:.2f} | {row['error_rate']*100:.3f} | {row['dropped_iterations']} | {row['db_median_ms']:.2f} | {row['app_median_ms']:.2f} | {row['db_time_share_median_pct']:.2f} | {row['db_queries_per_request']:.2f} |")
mixed_break_rate = Path(run_dir / "mixed/break_rate.txt").read_text().strip()
mixed_last_pass = int(Path(run_dir / "mixed/last_passed_rate.txt").read_text().strip())
print(f"Точка излома mixed: {mixed_break_rate}; допустимая интенсивность: {mixed_last_pass} req/s.")
operation_prefixes = {"GET /api/v1/loans": "op_list", "GET /api/v1/loans/{id}": "op_detail", "POST /api/v1/loans": "op_issue", "POST /api/v1/loans/{id}/return": "op_return", "GET /api/v1/summary": "op_summary"}
breakdown_rate = int(mixed_break_rate) if mixed_break_rate != "none" else mixed_last_pass
breakdown_summary = json.loads((run_dir / f"mixed/mixed_{breakdown_rate}.json").read_text())
operation_breakdown_rows = []
for operation, prefix in operation_prefixes.items():
    metrics = breakdown_summary["metrics"]
    request_count = metric_values(metrics, f"{prefix}_requests").get("count", 0)
    if request_count == 0:
        continue
    http_values = metric_values(metrics, f"{prefix}_http_ms")
    db_values = metric_values(metrics, f"{prefix}_db_ms")
    app_values = metric_values(metrics, f"{prefix}_app_ms")
    share_values = metric_values(metrics, f"{prefix}_db_share_pct")
    query_count = metric_values(metrics, f"{prefix}_db_queries").get("count", 0)
    operation_breakdown_rows.append({"operation": operation, "requests": request_count, "http_p50_ms": http_values.get("med", 0), "http_p95_ms": http_values.get("p(95)", 0), "http_max_ms": http_values.get("max", 0), "db_p50_ms": db_values.get("med", 0), "app_p50_ms": app_values.get("med", 0), "db_share_p50_pct": share_values.get("med", 0), "db_queries_per_request": query_count / request_count})
with (run_dir / "mixed/operation_breakdown.csv").open("w", newline="", encoding="utf-8") as target:
    writer = csv.DictWriter(target, fieldnames=["operation", "requests", "http_p50_ms", "http_p95_ms", "http_max_ms", "db_p50_ms", "app_p50_ms", "db_share_p50_pct", "db_queries_per_request"])
    writer.writeheader(); writer.writerows(operation_breakdown_rows)
print(f"Разложение по операциям на ступени {breakdown_rate} req/s:")
print("| Операция | Запросы | HTTP p50 ms | HTTP p95 ms | HTTP max ms | DB p50 ms | app p50 ms | DB share p50 % | DB queries/request |\n|---|---:|---:|---:|---:|---:|---:|---:|---:|")
for row in operation_breakdown_rows:
    print(f"| {row['operation']} | {row['requests']} | {row['http_p50_ms']:.2f} | {row['http_p95_ms']:.2f} | {row['http_max_ms']:.2f} | {row['db_p50_ms']:.2f} | {row['app_p50_ms']:.2f} | {row['db_share_p50_pct']:.2f} | {row['db_queries_per_request']:.2f} |")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([r["rate"] for r in mixed_rows], [r["p95_ms"] for r in mixed_rows], marker="o", label="mixed p95")
ax.axhline(500, color="red", linestyle="--", label="порог 500 ms")
ax.set(xlabel="Заданная интенсивность, req/s", ylabel="p95, ms", title="Mixed profile — p95 по ступеням")
ax.grid(True, alpha=0.3); ax.legend(); fig.tight_layout()
(repo_root / "reports").mkdir(exist_ok=True)
fig.savefig(repo_root / "reports/dz2_mixed_p95.png", dpi=160); plt.close(fig)
if mixed_break_rate != "none":
    resource_path = run_dir / f"resources/mixed_{mixed_break_rate}.csv"
    with resource_path.open(newline="", encoding="utf-8") as source:
        samples = list(csv.DictReader(source))
    print(f"Ресурсы ступени излома (полный CSV: {resource_path}): {len(samples)} samples")
    for key in ["app_cpu_pct_one_core", "ps_avg_cpu_pct", "app_rss_kb", "pg_cpu_pct", "db_connections", "db_active_connections"]:
        numeric = [float(s[key]) for s in samples if s.get(key, "") not in ("", "NA") and (key not in ("db_connections", "db_active_connections") or s[key].replace('.', '', 1).isdigit())]
        if numeric:
            print(f"{key}: median={statistics.median(numeric):.2f}, max={max(numeric):.2f}")
    print("Первые 3 и последние 3 сэмпла:")
    for sample in samples[:3] + samples[-3:]: print(sample)
else:
    print("Точка излома не достигнута в пределах запланированных ступеней.")

| Задано req/s | Фактически req/s | Медиана ms | p95 ms | Максимум ms | Ошибки % | dropped_iterations | DB median ms | app median ms | DB доля median % | DB queries/request |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 1.03 | 11.23 | 13.17 | 54.67 | 0.000 | 0 | 6.67 | 3.56 | 65.22 | 2.90 |
| 2 | 2.03 | 11.32 | 18.69 | 56.59 | 0.000 | 0 | 2.48 | 7.42 | 34.10 | 4.34 |
| 4 | 4.06 | 10.95 | 52.90 | 56.20 | 0.000 | 0 | 5.69 | 3.66 | 44.55 | 4.00 |
| 8 | 8.02 | 10.75 | 19.31 | 63.18 | 0.000 | 0 | 5.76 | 3.51 | 44.05 | 3.80 |
| 16 | 16.00 | 10.85 | 50.08 | 98.80 | 0.000 | 0 | 5.80 | 3.76 | 46.44 | 4.12 |
| 32 | 31.97 | 11.18 | 30.92 | 109.70 | 0.000 | 0 | 6.01 | 3.99 | 46.76 | 4.07 |
| 64 | 63.92 | 10.58 | 58.81 | 97.46 | 0.000 | 0 | 6.00 | 3.70 | 46.59 | 4.05 |
| 128 | 127.80 | 11.86 | 77.75 | 220.26 | 0.000 | 0 | 6.87 | 4.47 | 46.46 | 4.03 |
| 256 | 3.30 | 167.33 | 30153.59 | 30432.26 | 15.657 | 6504 | 11.71 | 131.88 | 8.70 | 3.59 |
Точка излома mixed: 256; допустимая инт

## Шаг 6. Тяжёлая операция — ступени `GET /api/v1/summary`
Перед наполнением large notebook-owned API останавливается, чтобы активные транзакции не блокировали `TRUNCATE`; после seed сервис запускается заново. Затем база засевается большим объёмом, чтобы опыт начинался с одинакового состояния. Выполняется та же лестница, пороги и схема охлаждения, что для mixed. Для heavy нет исчерпаемого пула ID. Точка излома определяется по первому threshold exit; если излома нет, фиксируется максимальная испытанная частота, без заявления о найденном насыщении.

In [9]:
%%bash
cd "$(git rev-parse --show-toplevel)"
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
RUN_DIR=$(cat logs/dz2/current_run.txt)
UVICORN_PID=$(cat "$RUN_DIR/uvicorn.pid")
if kill -0 "$UVICORN_PID" 2>/dev/null; then kill "$UVICORN_PID"; fi
stopped=0
for attempt in $(seq 1 15); do
  state=$(ps -p "$UVICORN_PID" -o stat= 2>/dev/null || true)
  if [ -z "$state" ] || [[ "$state" == Z* ]]; then stopped=1; break; fi
  sleep 1
done
if [ "$stopped" -ne 1 ]; then kill -9 "$UVICORN_PID"; fi
if curl --fail --silent --max-time 1 http://localhost:8000/docs >/dev/null; then echo 'Old uvicorn still responds after restart request.' >&2; exit 1; fi
.venv/bin/python seed.py --type=large | sed -E 's#(Login: admin / ).*#\1[REDACTED]#'
nohup .venv/bin/uvicorn app.main:app --port 8000 > "$RUN_DIR/uvicorn_heavy.log" 2>&1 &
UVICORN_PID=$!
echo "$UVICORN_PID" > "$RUN_DIR/uvicorn.pid"
ready=0
for attempt in $(seq 1 30); do
  echo "API readiness for heavy ${attempt}/30"
  if curl --fail --silent --max-time 2 http://localhost:8000/docs >/dev/null; then ready=1; break; fi
  sleep 1
done
if [ "$ready" -ne 1 ]; then cat "$RUN_DIR/uvicorn_heavy.log"; kill "$UVICORN_PID" 2>/dev/null || true; exit 1; fi
read_ticks() { awk '{sub(/^.*\) /, ""); print $12 + $13}' "/proc/$UVICORN_PID/stat"; }
CLK_TCK=$(getconf CLK_TCK)
last_rate=0
break_rate=none
for rate in 1 2 4 8 16 32 64 128 256 512; do
  echo "=== heavy summary: RATE=$rate req/s, DURATION=30s ==="
  json="$RUN_DIR/heavy/heavy_${rate}.json"; log="$RUN_DIR/heavy/heavy_${rate}.log"
  resource="$RUN_DIR/resources/heavy_${rate}.csv"; stopfile="$RUN_DIR/resources/stop_heavy_${rate}"
  rm -f "$stopfile"
  printf 'timestamp,phase,rate,app_cpu_pct_one_core,ps_avg_cpu_pct,app_rss_kb,pg_cpu_pct,pg_mem_usage,db_connections,db_active_connections\n' > "$resource"
  (
    previous=$(read_ticks)
    while [ ! -e "$stopfile" ]; do
      sleep 1; current=$(read_ticks)
      app_cpu=$(awk -v d="$((current-previous))" -v hz="$CLK_TCK" 'BEGIN { printf "%.2f", d*100/hz }'); previous=$current
      ps_values=$(ps -p "$UVICORN_PID" -o %cpu=,rss= | awk '{printf "%s,%s", $1, $2}')
      pg_stats=$(docker stats --no-stream --format '{{.CPUPerc}},{{.MemUsage}}' ivan_bashkin_postgres | awk -F, '{gsub(/%/, "", $1); gsub(/ /, "", $2); printf "%s,%s", $1, $2}')
      db_conn=$(docker compose exec -T ivan_bashkin_postgres psql -X -qAt -U app -d ivan_bashkin -c "SELECT count(*) || ',' || count(*) FILTER (WHERE state='active') FROM pg_stat_activity WHERE datname=current_database()" </dev/null | tr -d '\r\n')
      printf '%s,heavy_%s,%s,%s,%s,%s,%s\n' "$(date -Is)" "$rate" "$rate" "$app_cpu" "$ps_values" "$pg_stats" "$db_conn" >> "$resource"
    done
  ) & monitor_pid=$!
  set +e
  k6 run -e SCENARIO=heavy -e RATE="$rate" -e DURATION=30s -e DATA_DIR="$PWD/$RUN_DIR/ids" --summary-export "$json" load/library.js > "$log" 2>&1
  k6_status=$?
  set -e
  if [ ! -f "$json" ]; then touch "$stopfile"; wait "$monitor_pid" || true; cat "$log"; exit 1; fi
  echo '--- Full k6 output ---'; cat "$log"; echo "k6 exit status: $k6_status"
  if [ "$k6_status" -eq 99 ]; then break_rate=$rate; touch "$stopfile"; wait "$monitor_pid"; echo "Heavy threshold violated at $rate req/s; stopping."; break
  elif [ "$k6_status" -ne 0 ]; then touch "$stopfile"; wait "$monitor_pid" || true; echo "Unexpected k6 failure $k6_status" >&2; exit "$k6_status"; fi
  stable=0; previous=$(read_ticks)
  for second in $(seq 1 180); do
    sleep 1; current=$(read_ticks)
    idle_cpu=$(awk -v d="$((current-previous))" -v hz="$CLK_TCK" 'BEGIN { printf "%.2f", d*100/hz }'); previous=$current
    if awk -v cpu="$idle_cpu" 'BEGIN { exit !(cpu < 5) }'; then stable=$((stable+1)); else stable=0; fi
    if [ "$stable" -ge 5 ]; then echo "CPU idle recovered after ${second}s."; break; fi
  done
  touch "$stopfile"; wait "$monitor_pid"
  if [ "$stable" -lt 5 ]; then echo 'CPU did not return to idle within 180 seconds; stopping.' >&2; exit 1; fi
  last_rate=$rate
done
printf '%s\n' "$break_rate" > "$RUN_DIR/heavy/break_rate.txt"
printf '%s\n' "$last_rate" > "$RUN_DIR/heavy/last_passed_rate.txt"
echo "Heavy staircase finished; first failed rate=$break_rate, last passing rate=$last_rate."

Seeded 3000 readers, 10000 books, 30000 copies, 75000 loans. Login: admin / [REDACTED]


API readiness for heavy 1/30


API readiness for heavy 2/30


=== heavy summary: RATE=1 req/s, DURATION=30s ===


--- Full k6 output ---



         /\      Grafana   /‾‾/  
    /\  /  \     |\  __   /  /   
   /  \/    \    | |/ /  / 

... (611 outputs left out)

on metrics 'http_req_duration, http_req_failed' have been crossed"


k6 exit status: 99


Heavy threshold violated at 64 req/s; stopping.


Heavy staircase finished; first failed rate=64, last passing rate=32.


### Heavy: таблица ступеней и выбор частоты профилирования
Строится таблица метрик из summary JSON. При наличии излома профиль снимается на первой не прошедшей ступени; иначе на максимальной испытанной ступени. CSV хранит все значения.

In [10]:
heavy_rows = read_stages("heavy")
with (run_dir / "heavy/stages.csv").open("w", newline="", encoding="utf-8") as target:
    writer = csv.DictWriter(target, fieldnames=["rate", "actual_rate", "count", "median_ms", "p95_ms", "max_ms", "error_rate", "dropped_iterations", "db_median_ms", "db_p95_ms", "app_median_ms", "app_p95_ms", "db_time_share_median_pct", "db_queries_per_request"])
    writer.writeheader(); writer.writerows(heavy_rows)
print("| Задано req/s | Фактически req/s | Медиана ms | p95 ms | Максимум ms | Ошибки % | dropped_iterations | DB median ms | app median ms | DB доля median % | DB queries/request |\n|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
for row in heavy_rows:
    print(f"| {row['rate']} | {row['actual_rate']:.2f} | {row['median_ms']:.2f} | {row['p95_ms']:.2f} | {row['max_ms']:.2f} | {row['error_rate']*100:.3f} | {row['dropped_iterations']} | {row['db_median_ms']:.2f} | {row['app_median_ms']:.2f} | {row['db_time_share_median_pct']:.2f} | {row['db_queries_per_request']:.2f} |")
heavy_break_rate = Path(run_dir / "heavy/break_rate.txt").read_text().strip()
heavy_last_pass = int(Path(run_dir / "heavy/last_passed_rate.txt").read_text().strip())
profile_rate = int(heavy_break_rate) if heavy_break_rate != "none" else heavy_rows[-1]["rate"]
(run_dir / "heavy/profile_rate.txt").write_text(str(profile_rate))
print(f"Heavy излом: {heavy_break_rate}; допустимая частота: {heavy_last_pass}; частота профилирования: {profile_rate} req/s.")
if heavy_break_rate == "none":
    print("Порог не нарушен в пределах испытанной лестницы; насыщение пропускной способности не объявляется без подтверждения по фактической частоте и dropped_iterations.")
else:
    failed = next(r for r in heavy_rows if r["rate"] == int(heavy_break_rate))
    if not heavy_rows or failed["rate"] == heavy_rows[0]["rate"]:
        print("Первую ступень порог не прошла: точки излома по заданному правилу нет; анализируется насыщение по фактической частоте и dropped_iterations.")

| Задано req/s | Фактически req/s | Медиана ms | p95 ms | Максимум ms | Ошибки % | dropped_iterations | DB median ms | app median ms | DB доля median % | DB queries/request |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 1.03 | 57.87 | 85.69 | 112.22 | 0.000 | 0 | 52.85 | 3.95 | 93.02 | 3.87 |
| 2 | 2.06 | 56.25 | 78.22 | 94.52 | 0.000 | 0 | 51.66 | 3.86 | 92.96 | 3.94 |
| 4 | 4.03 | 58.48 | 81.54 | 118.30 | 0.000 | 0 | 53.74 | 3.93 | 93.19 | 3.97 |
| 8 | 8.04 | 56.25 | 72.06 | 154.18 | 0.000 | 0 | 51.51 | 3.88 | 93.14 | 3.98 |
| 16 | 16.00 | 54.28 | 65.56 | 99.76 | 0.000 | 0 | 49.45 | 3.89 | 92.78 | 3.99 |
| 32 | 31.94 | 90.35 | 143.30 | 212.57 | 0.000 | 0 | 83.89 | 5.45 | 93.65 | 4.00 |
| 64 | 23.49 | 113.95 | 862.69 | 30376.87 | 1.772 | 294 | 103.95 | 6.66 | 93.46 | 3.93 |
Heavy излом: 64; допустимая частота: 32; частота профилирования: 64 req/s.


## Шаг 7. Профилирование тяжёлой операции (py-spy)
На выбранной частоте одновременно 30 секунд идёт heavy k6, а py-spy пытается снять обязательные 15-секундные flamegraph и raw профили процесса uvicorn. Ошибка ptrace не скрывается: полные сообщения сохраняются в `logs/.../profiles/`; в случае успеха SVG и raw будут в `profiles/`.

In [11]:
%%bash
cd "$(git rev-parse --show-toplevel)"
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
RUN_DIR=$(cat logs/dz2/current_run.txt)
RATE=$(cat "$RUN_DIR/heavy/profile_rate.txt")
UVICORN_PID=$(cat "$RUN_DIR/uvicorn.pid")
mkdir -p profiles "$RUN_DIR/profiles"
set +e
.venv/bin/py-spy record -p "$UVICORN_PID" -d 15 -f flamegraph -o "$RUN_DIR/profiles/summary_flame.svg" > "$RUN_DIR/profiles/flame.log" 2>&1 & flame_pid=$!
.venv/bin/py-spy record -p "$UVICORN_PID" -d 15 -f raw -o "$RUN_DIR/profiles/summary_raw.txt" > "$RUN_DIR/profiles/raw.log" 2>&1 & raw_pid=$!
k6 run -e SCENARIO=heavy -e RATE="$RATE" -e DURATION=30s -e DATA_DIR="$PWD/$RUN_DIR/ids" --summary-export "$RUN_DIR/heavy/profile_run.json" load/library.js > "$RUN_DIR/heavy/profile_run.log" 2>&1
k6_status=$?
wait "$flame_pid"; flame_status=$?
wait "$raw_pid"; raw_status=$?
set -e
echo "k6 profile run status=$k6_status; py-spy flamegraph status=$flame_status; py-spy raw status=$raw_status"
echo '--- py-spy flamegraph output ---'; cat "$RUN_DIR/profiles/flame.log"
echo '--- py-spy raw output ---'; cat "$RUN_DIR/profiles/raw.log"
echo '--- k6 profile run output ---'; cat "$RUN_DIR/heavy/profile_run.log"
if [ "$flame_status" -eq 0 ] && [ -f "$RUN_DIR/profiles/summary_flame.svg" ]; then echo "Flamegraph saved: $RUN_DIR/profiles/summary_flame.svg"; else echo 'Flamegraph not captured; inspect py-spy error for ptrace/platform restriction.'; fi
if [ "$raw_status" -eq 0 ] && [ -f "$RUN_DIR/profiles/summary_raw.txt" ]; then echo "Raw profile saved: $RUN_DIR/profiles/summary_raw.txt"; else echo 'Raw profile not captured; inspect py-spy error for ptrace/platform restriction.'; fi

k6 profile run status=100; py-spy flamegraph status=1; py-spy raw status=1
--- py-spy flamegraph out

put ---


Permission Denied: Try running again with elevated permissions by going 'sudo env "PATH=$PATH" !!'


--- py-spy raw output ---


Permission Denied: Try running again with elevated permissions by going 'sudo env "PATH=$PATH" !!'


--- k6 profile run output ---


... (36 outputs left out)

:27-03:00" level=error msg="setup() execution timed out after 60 seconds" hint="You can increase the

 time limit via the setupTimeout option"


Flamegraph not captured; inspect py-spy error for ptrace/platform restriction.
Raw profile not captu

red; inspect py-spy error for ptrace/platform restriction.


## Шаг 8. EXPLAIN ANALYZE, одновременная выдача и профиль функций
Исполняемые планы сохраняются полностью в `logs/.../explain_plans.txt`; SELECT-планы запускаются в read-only транзакции. UPDATE/INSERT планы из семи запросов POST запускаются через `EXPLAIN ANALYZE` внутри транзакции, которая немедленно откатывается. Затем проверяется, что активные выдачи не дублируют экземпляры. При наличии raw py-spy строится таблица горячих Python-фреймов и долей samples.

In [12]:
%%bash
cd "$(git rev-parse --show-toplevel)"
set -euo pipefail
RUN_DIR=$(cat logs/dz2/current_run.txt)
cat > "$RUN_DIR/explain.sql" <<'SQL'
BEGIN READ ONLY;
EXPLAIN (ANALYZE, BUFFERS) SELECT count(book_copies.id) FROM book_copies WHERE book_copies.status = 'issued';
EXPLAIN (ANALYZE, BUFFERS) SELECT count(loans.id) FROM loans WHERE loans.return_date IS NULL AND loans.due_date < now();
EXPLAIN (ANALYZE, BUFFERS) SELECT books.id, books.title, books.author, count(loans.id) AS loan_count FROM books JOIN book_copies ON book_copies.book_id = books.id JOIN loans ON loans.copy_id = book_copies.id GROUP BY books.id, books.title, books.author ORDER BY count(loans.id) DESC, books.id LIMIT 10;
EXPLAIN (ANALYZE, BUFFERS) SELECT count(loans.id) FROM loans WHERE loans.return_date IS NULL;
EXPLAIN (ANALYZE, BUFFERS) SELECT loans.id, loans.reader_id, loans.copy_id, loans.issue_date, loans.due_date, loans.return_date, loans.penalty_amount FROM loans ORDER BY loans.id LIMIT 20 OFFSET 0;
EXPLAIN (ANALYZE, BUFFERS) SELECT loans.id, loans.reader_id, loans.copy_id, loans.issue_date, loans.due_date, loans.return_date, loans.penalty_amount, readers_1.id, readers_1.full_name, readers_1.ticket_number, books_1.id, books_1.title, books_1.author, books_1.isbn, books_1.category, book_copies_1.id, book_copies_1.book_id, book_copies_1.inventory_number, book_copies_1.status FROM loans LEFT OUTER JOIN readers AS readers_1 ON readers_1.id = loans.reader_id LEFT OUTER JOIN book_copies AS book_copies_1 ON book_copies_1.id = loans.copy_id LEFT OUTER JOIN books AS books_1 ON books_1.id = book_copies_1.book_id WHERE loans.id = 1;
EXPLAIN (ANALYZE, BUFFERS) SELECT users.id, users.username, users.password_hash, users.role FROM users WHERE users.username = 'admin';
COMMIT;
BEGIN;
EXPLAIN (ANALYZE, BUFFERS) SELECT users.id, users.username, users.password_hash, users.role FROM users WHERE users.id = 1;
EXPLAIN (ANALYZE, BUFFERS) SELECT readers.id, readers.full_name, readers.ticket_number, readers.created_at FROM readers WHERE readers.id = 1;
EXPLAIN (ANALYZE, BUFFERS) SELECT book_copies.id, book_copies.book_id, book_copies.inventory_number, book_copies.status FROM book_copies WHERE book_copies.id = 1;
EXPLAIN (ANALYZE, BUFFERS) SELECT loans.id FROM loans WHERE loans.reader_id = 1 AND loans.return_date IS NULL AND loans.due_date < now() LIMIT 1;
EXPLAIN (ANALYZE, BUFFERS) UPDATE book_copies SET status = 'issued' WHERE id = 1;
EXPLAIN (ANALYZE, BUFFERS) INSERT INTO loans (reader_id, copy_id, issue_date, due_date, return_date, penalty_amount) VALUES (1, 1, now(), now() + interval '14 days', NULL, 0);
EXPLAIN (ANALYZE, BUFFERS) SELECT loans.id, loans.reader_id, loans.copy_id, loans.issue_date, loans.due_date, loans.return_date, loans.penalty_amount FROM loans WHERE loans.id = 1;
ROLLBACK;
BEGIN;
EXPLAIN (ANALYZE, BUFFERS) SELECT loans.id, loans.reader_id, loans.copy_id, loans.issue_date, loans.due_date, loans.return_date, loans.penalty_amount FROM loans WHERE loans.id = 4;
EXPLAIN (ANALYZE, BUFFERS) UPDATE loans SET return_date = now(), penalty_amount = 0 WHERE id = 4;
EXPLAIN (ANALYZE, BUFFERS) UPDATE book_copies SET status = 'available' WHERE id = 4;
ROLLBACK;
SQL
docker compose exec -T ivan_bashkin_postgres psql -X -v ON_ERROR_STOP=1 -U app -d ivan_bashkin -f - < "$RUN_DIR/explain.sql" > "$RUN_DIR/explain_plans.txt"
echo 'Complete EXPLAIN output:'
cat "$RUN_DIR/explain_plans.txt"
docker compose exec -T ivan_bashkin_postgres psql -X -qAt -v ON_ERROR_STOP=1 -U app -d ivan_bashkin -c "BEGIN READ ONLY; SELECT copy_id, count(*) FROM loans WHERE return_date IS NULL GROUP BY copy_id HAVING count(*) > 1; COMMIT;" </dev/null > "$RUN_DIR/heavy/duplicate_active_copies.txt"
if [ -s "$RUN_DIR/heavy/duplicate_active_copies.txt" ]; then echo 'Duplicate active copy assignments:' >&2; cat "$RUN_DIR/heavy/duplicate_active_copies.txt"; exit 1; fi
echo 'No duplicate active copy assignments after all tests.'
if [ -f "$RUN_DIR/profiles/summary_raw.txt" ]; then echo 'Raw py-spy profile captured.'; else echo 'py-spy raw profile missing; profile step has explicit ptrace status above.'; fi

Complete EXPLAIN output:


BEGIN
                                                    QUERY PLAN                                

                     
------------------------------------------------------------------------------

-------------------------------------
 Aggregate  (cost=614.75..614.76 rows=1 width=8) (actual time=

2.632..2.635 rows=1 loops=1)
   Buffers: shared hit=221
   ->  Seq Scan on book_copies  (cost=0.00..

596.00 rows=7500 width=4) (actual time=0.005..2.317 rows=7500 loops=1)
         Filter: ((status)::t

... (124 outputs left out)

=64) (actual time=0.010..0.011 rows=1 loops=1)
         Index Cond: (id = 4)
         Buffers: share

d hit=3
 Planning Time: 0.024 ms
 Execution Time: 0.029 ms
(7 rows)

ROLLBACK


No duplicate active copy assignments after all tests.


py-spy raw profile missing; profile step has explicit ptrace status above.


### Анализ raw profile и resource samples
Если raw-профиль создан, частоты стеков агрегируются по leaf-функциям и показываются горячие символы с долями. Для ступени тяжелой операции показываются min/median/max ресурсов и ограниченный фрагмент CSV; полный временной ряд остаётся в `logs/`.

In [13]:
from collections import Counter
import csv
import re
import statistics

raw_path = run_dir / "profiles/summary_raw.txt"
if raw_path.exists():
    counts = Counter()
    total = 0
    for line in raw_path.read_text(errors="replace").splitlines():
        match = re.match(r"^\s*(.*?)\s+(\d+)\s*$", line)
        if not match:
            continue
        stack, samples = match.group(1), int(match.group(2))
        total += samples
        leaf = stack.split(";")[-1]
        counts[leaf] += samples
    print(f"Raw sample count: {total}")
    print("| Горячий leaf frame | Samples | Доля % |\n|---|---:|---:|")
    for name, count in counts.most_common(20):
        print(f"| `{name}` | {count} | {100*count/total:.2f} |")
else:
    print("Не выполнено: py-spy raw профиль не создан; см. logs/.../profiles/raw.log для причины.")

if heavy_break_rate != "none":
    heavy_resource = run_dir / f"resources/heavy_{heavy_break_rate}.csv"
else:
    heavy_resource = run_dir / f"resources/heavy_{heavy_rows[-1]['rate']}.csv"
if heavy_resource.exists():
    with heavy_resource.open(newline="", encoding="utf-8") as source:
        resource_rows = list(csv.DictReader(source))
    print(f"Ресурсы heavy на {heavy_resource.stem}: {len(resource_rows)} samples; полный ряд: {heavy_resource}")
    for column in ["app_cpu_pct_one_core", "ps_avg_cpu_pct", "app_rss_kb", "pg_cpu_pct", "db_connections", "db_active_connections"]:
        values = [float(row[column]) for row in resource_rows if row.get(column, "").replace('.', '', 1).isdigit()]
        if values:
            print(f"{column}: min={min(values):.2f}, median={statistics.median(values):.2f}, max={max(values):.2f}")
    print("Первый/последний samples:")
    for row in resource_rows[:2] + resource_rows[-2:]: print(row)

Не выполнено: py-spy raw профиль не создан; см. logs/.../profiles/raw.log для причины.
Ресурсы heavy на heavy_64: 27 samples; полный ряд: /home/ewvn/MISIS/OKSP/kr1/ivan_bashkin/logs/dz2/20261120T205841Z/resources/heavy_64.csv
app_cpu_pct_one_core: min=0.00, median=5.00, max=148.00
ps_avg_cpu_pct: min=6.10, median=9.10, max=9.90
app_rss_kb: min=101360.00, median=114108.00, max=117444.00
pg_cpu_pct: min=0.05, median=5.38, max=903.26
db_connections: min=10.00, median=16.00, max=16.00
db_active_connections: min=1.00, median=1.00, max=13.00
Первый/последний samples:
{'timestamp': '2026-11-20T18:09:27-03:00', 'phase': 'heavy_64', 'rate': '64', 'app_cpu_pct_one_core': '51.00', 'ps_avg_cpu_pct': '6.1', 'app_rss_kb': '101360', 'pg_cpu_pct': '899.06', 'pg_mem_usage': '101.8MiB/14.95GiB', 'db_connections': '13', 'db_active_connections': '9'}
{'timestamp': '2026-11-20T18:09:29-03:00', 'phase': 'heavy_64', 'rate': '64', 'app_cpu_pct_one_core': '141.00', 'ps_avg_cpu_pct': '6.7', 'app_rss_kb': '10189

## Шаг 9. График heavy, гипотезы ДЗ1 и направление ДЗ3
Сохраняются PNG p95 по heavy-ступеням. Вердикты формируются только по фактическим p50/p95, доле БД в измерении ответа, EXPLAIN планам и raw профилю; если данных для доказательства нет, вердикт указывается как «не проверена». Дополнительный опыт с workers не проводился, сервисная конфигурация не менялась.

In [14]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([r["rate"] for r in heavy_rows], [r["p95_ms"] for r in heavy_rows], marker="o", color="darkorange", label="heavy summary p95")
ax.axhline(500, color="red", linestyle="--", label="порог 500 ms")
ax.set(xlabel="Заданная интенсивность, req/s", ylabel="p95, ms", title="Heavy summary — p95 по ступеням")
ax.grid(True, alpha=0.3); ax.legend(); fig.tight_layout()
fig.savefig(repo_root / "reports/dz2_heavy_p95.png", dpi=160); plt.close(fig)

summary_row = next((r for r in heavy_rows if r["rate"] == profile_rate), heavy_rows[-1])
db_fraction = summary_row["db_time_share_median_pct"]
print(f"Профилируемая ступень: HTTP p50={summary_row['median_ms']:.2f} ms, DB p50={summary_row['db_median_ms']:.2f} ms, app p50={summary_row['app_median_ms']:.2f} ms, медиана доли DB в серверном времени={db_fraction:.2f}%, запросов к БД на HTTP-запрос={summary_row['db_queries_per_request']:.2f}.")
if raw_path.exists():
    profile_counts = Counter()
    profile_total = 0
    for line in raw_path.read_text(errors="replace").splitlines():
        m = re.match(r"^\s*(.*?)\s+(\d+)\s*$", line)
        if m:
            profile_counts[m.group(1).split(";")[-1]] += int(m.group(2)); profile_total += int(m.group(2))
    service_frames = sum(n for frame, n in profile_counts.items() if "get_summary" in frame or "summary" in frame.lower())
    if profile_total:
        print(f"Heavy sampled frames: {profile_total}; frames naming summary/get_summary: {service_frames} ({100*service_frames/profile_total:.2f}%). SQL wait is measured separately by request timing headers.")

print("## Гипотезы ДЗ1")
print(f"1. GET /api/v1/summary на {summary_row['rate']} req/s: HTTP p95={summary_row['p95_ms']:.2f} ms, DB p50={summary_row['db_median_ms']:.2f} ms, app p50={summary_row['app_median_ms']:.2f} ms, медианная доля DB в серверном времени={db_fraction:.2f}%; планы: {run_dir / 'explain_plans.txt'}. Узкое место Python оценивается отдельно по валидному py-spy профилю.")
list_rows = mixed_rows
if list_rows:
    list_metric = next((row for row in operation_breakdown_rows if row["operation"] == "GET /api/v1/loans"), None)
    if list_metric:
        print(f"2. GET /api/v1/loans на mixed {breakdown_rate} req/s: HTTP p95={list_metric['http_p95_ms']:.2f} ms, DB p50={list_metric['db_p50_ms']:.2f} ms, app p50={list_metric['app_p50_ms']:.2f} ms, DB доля p50={list_metric['db_share_p50_pct']:.2f}%, запросов к БД на операцию={list_metric['db_queries_per_request']:.2f}; запросы списка и COUNT проверены в {run_dir / 'explain_plans.txt'}.")
    else:
        print("2. GET /api/v1/loans: operation-level k6 sample отсутствует на выбранной ступени.")
issue_metric = next((row for row in operation_breakdown_rows if row["operation"] == "POST /api/v1/loans"), None)
if issue_metric:
    print(f"3. POST /api/v1/loans на mixed {breakdown_rate} req/s: HTTP p95={issue_metric['http_p95_ms']:.2f} ms, DB p50={issue_metric['db_p50_ms']:.2f} ms, app p50={issue_metric['app_p50_ms']:.2f} ms, DB доля p50={issue_metric['db_share_p50_pct']:.2f}%; семь SQL действий плана в {run_dir / 'explain_plans.txt'}. Изменение относительно baseline не проверено.")
else:
    print("3. POST /api/v1/loans: operation-level k6 sample отсутствует на выбранной ступени.")
print("Направление ДЗ3: исходя из фактически тяжёлых операций и EXPLAIN/raw-профиля, оптимизировать только подтверждённое узкое место; см. таблицы и планы выше. Дополнительный опыт с --workers 2 не проводился.")

Профилируемая ступень: HTTP p50=113.95 ms, DB p50=103.95 ms, app p50=6.66 ms, медиана доли DB в серверном времени=93.46%, запросов к БД на HTTP-запрос=3.93.
## Гипотезы ДЗ1
1. GET /api/v1/summary на 64 req/s: HTTP p95=862.69 ms, DB p50=103.95 ms, app p50=6.66 ms, медианная доля DB в серверном времени=93.46%; планы: /home/ewvn/MISIS/OKSP/kr1/ivan_bashkin/logs/dz2/20261120T205841Z/explain_plans.txt. Узкое место Python оценивается отдельно по валидному py-spy профилю.
2. GET /api/v1/loans на mixed 256 req/s: HTTP p95=30250.13 ms, DB p50=11.13 ms, app p50=136.55 ms, DB доля p50=7.35%, запросов к БД на операцию=3.00; запросы списка и COUNT проверены в /home/ewvn/MISIS/OKSP/kr1/ivan_bashkin/logs/dz2/20261120T205841Z/explain_plans.txt.
3. POST /api/v1/loans на mixed 256 req/s: HTTP p95=390.87 ms, DB p50=19.63 ms, app p50=93.05 ms, DB доля p50=16.99%; семь SQL действий плана в /home/ewvn/MISIS/OKSP/kr1/ivan_bashkin/logs/dz2/20261120T205841Z/explain_plans.txt. Изменение относительно baseline не

## Шаг 10. Остановка сервиса и сохранение состояния
Uvicorn останавливается по PID. Проверяется, что `/docs` больше не отвечает; PostgreSQL остаётся запущенным. Сохраняются compose status и полный список файлов, относящихся к испытанию.

In [15]:
%%bash
cd "$(git rev-parse --show-toplevel)"
set -euo pipefail
RUN_DIR=$(cat logs/dz2/current_run.txt)
pid=$(cat "$RUN_DIR/uvicorn.pid")
if kill -0 "$pid" 2>/dev/null; then kill "$pid"; fi
stopped=0
for attempt in $(seq 1 15); do
  state=$(ps -p "$pid" -o stat= 2>/dev/null || true)
  if [ -z "$state" ] || [[ "$state" == Z* ]]; then stopped=1; break; fi
  sleep 1
done
if [ "$stopped" -ne 1 ]; then kill -9 "$pid"; fi
port_free=0
for attempt in $(seq 1 10); do
  if ! curl --fail --silent --max-time 1 http://localhost:8000/docs >/dev/null; then port_free=1; break; fi
  sleep 1
done
if [ "$port_free" -ne 1 ]; then echo 'uvicorn still responds on port 8000' >&2; exit 1; fi
echo "Stopped uvicorn PID $pid; port 8000 no longer responds."
docker compose ps | tee "$RUN_DIR/docker-compose-ps.log"
find "$RUN_DIR" profiles reports load -type f -print | sort | tee "$RUN_DIR/files.txt"
echo "Full environment: logs/environment.txt"
echo "Run outputs: $RUN_DIR"

Stopped uvicorn PID 176655; port 8000 no longer responds.


NAME                    IMAGE         COMMAND                  SERVICE                 CREATED      

 STATUS                 PORTS
ivan_bashkin_postgres   postgres:16   "docker-entrypoint.s…"   ivan_

bashkin_postgres   2 hours ago   Up 2 hours (healthy)   0.0.0.0:5432->5432/tcp, [::]:5432->5432/tcp


load/library.js
logs/dz2/20261120T205841Z/docker-compose-ps.log
logs/dz2/20261120T205841Z/explain_pl

ans.txt
logs/dz2/20261120T205841Z/explain.sql
logs/dz2/20261120T205841Z/heavy/break_rate.txt
logs/dz

... (39 outputs left out)

/20261120T205841Z/smoke/mixed.json
logs/dz2/20261120T205841Z/smoke/mixed.log
logs/dz2/20261120T20584

1Z/uvicorn_heavy.log
logs/dz2/20261120T205841Z/uvicorn.log
logs/dz2/20261120T205841Z/uvicorn.pid
rep

orts/dz2_heavy_p95.png
reports/dz2_mixed_p95.png


Full environment: logs/environment.txt
Run outputs: logs/dz2/20261120T205841Z
